# Cosmic Census
## Classificação de Objetos Celestes com Big Data

### Sobre o projeto
#### Sloan Digital Sky Survey (SDSS) | PySpark | Pandas | Plotly | TensorFlow

O **Sloan Digital Sky Survey (SDSS)** é um dos maiores levantamentos astronômicos da história,
tendo catalogado mais de 500 milhões de objetos celestes desde o ano 2000.
Neste projeto usamos 100 mil observações reais do SDSS para construir um **pipeline completo de Big Data**
que responde à pergunta científica:

> *"Dado um conjunto de medições fotométricas de um objeto no céu, ele é uma estrela, uma galáxia ou um quasar (QSO)?"*

### Glossário do domínio astronômico

| Termo | Significado |
|---|---|
| **Galáxia** | Sistema de bilhões de estrelas ligadas pela gravidade (ex: Via Láctea) |
| **Estrela** | Corpo celeste em fusão nuclear; as do dataset estão na nossa própria galáxia |
| **Quasar (QSO)** | "Quasi-Stellar Object" — núcleo galáctico ativo extremamente luminoso e distante |
| **u, g, r, i, z** | Os 5 filtros fotométricos do SDSS: u=ultravioleta, g=verde, r=vermelho, i=infravermelho próximo, z=infravermelho. O telescópio fotografa cada objeto 5 vezes, uma por filtro, gerando 5 valores de brilho (magnitudes) |
| **Magnitude** | Medida de brilho de um objeto: quanto maior o número, mais fraco o objeto |
| **Redshift (z)** | Desvio para o vermelho da luz — indica distância. Estrelas: z ≈ 0 (perto). Galáxias: z ~ 0.5. Quasares: z > 1 (os mais distantes já observados) |
| **Índice de cor** | Diferença entre magnitudes em dois filtros (ex: u−g). Objetos de tipos diferentes ocupam regiões distintas no espaço de cores — é a "impressão digital" espectral de cada tipo |

### O que é Machine Learning aqui?

O algoritmo **não é programado com regras**. Em vez disso, você mostra milhares de exemplos já classificados
por astrônomos (estrelas, galáxias, quasares) e o algoritmo descobre sozinho quais padrões de brilho e cor
separam as três classes. Isso se chama **aprendizado supervisionado**.

O **Random Forest** é um dos algoritmos mais usados: ele cria centenas de "árvores de decisão",
cada uma treinada num subconjunto aleatório dos dados, e faz votação no final.
A "floresta" de árvores é muito mais robusta do que uma árvore só.

### Conteúdos da disciplina cobertos

| Módulo | O que aplicamos |
|---|---|
| Princípios de Big Data | Os 5 Vs, ecossistema, data lake |
| Hadoop e armazenamento | HDFS vs RDBMS, Parquet como Data Lake |
| PySpark | SparkSession, RDDs, DataFrames, MapReduce, transformações |
| Pandas + Plotly | EDA completa, 5 visualizações interativas, mapa celeste |
| Big Data Analytics | KDD pipeline, Scikit-learn Random Forest, rede neural com TensorFlow/Keras |

### Arquitetura do pipeline

```
[SDSS CSV]  ->  [Data Lake Parquet]  ->  [PySpark: limpeza + features]
    ->  [Pandas: EDA + Plotly]  ->  [TensorFlow: classificação]
    ->  [Avaliação + Spark MLlib Pipeline]
```


---
## 0. Instalação de dependências

Execute esta célula uma única vez.
- **Google Colab**: tudo já instalado, exceto `pyspark`
- **VSCode / Jupyter local**: precisa do **Java 8+** no PATH (necessário para o Spark)


In [ ]:
import subprocess, sys

pacotes = [
    "pyspark",       # Apache Spark para Python
    "pandas",        # Analise de dados
    "numpy",         # Operacoes numericas
    "plotly",        # Visualizacoes interativas
    "scikit-learn",  # Machine Learning classico
    "tensorflow",    # Redes neurais profundas
    "requests",      # Download do dataset
    "pyarrow",       # Suporte a Parquet (Data Lake)
]

for p in pacotes:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', p, '-q'])

print("Todas as dependencias instaladas!")


---
## 1. Imports e configuração global

Boa prática: centralizar todos os imports em uma única célula no topo do notebook.


In [ ]:
import os, warnings, requests, time, pickle, json
from datetime import datetime
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

# PySpark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler, StandardScaler as SparkScaler
from pyspark.ml.classification import RandomForestClassifier as SparkRFC
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline

# Visualizacao
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Machine Learning classico
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier as RFC
from sklearn.metrics import classification_report, confusion_matrix

# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print(f"PySpark    {__import__('pyspark').__version__}")
print(f"Pandas     {pd.__version__}")
print(f"TensorFlow {tf.__version__}")

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Paleta cromatica astronomica
CORES = {"GALAXY": "#5B8FD4", "STAR": "#F5C842", "QSO": "#E05C5C"}


---
## 2. Princípios de Big Data — contexto e aquisição dos dados

### Os 5 Vs do Big Data aplicados ao SDSS

| V | No SDSS |
|---|---|
| **Volume** | Mais de 500 milhões de objetos, ~60 TB de imagens brutas |
| **Velocidade** | Novas observações toda noite de observação |
| **Variedade** | Dados fotométricos (u,g,r,i,z), espectroscópicos, imagens FITS |
| **Veracidade** | Medições com incerteza instrumental calibrada |
| **Valor** | Descoberta de quasares, galáxias distantes, matéria escura |

### Conceito de Data Lake

Ingerimos os dados brutos e armazenamos em **Parquet** (formato colunar comprimido).
Em produção este arquivo estaria no **HDFS** ou Amazon S3. Aqui simulamos o mesmo
padrão arquitetural localmente.

```
[Fonte: SDSS CSV]  ->  [Data Lake: ./data/sdss_lake.parquet]
                                  |
                    +-------------+
                    |   PySpark   |   processamento distribuído
                    +-------------+
```


In [ ]:
import os, requests
import pandas as pd

os.makedirs("data", exist_ok=True)
CSV     = "data/sdss_raw.csv"
PARQUET = "data/sdss_lake.parquet"

if not os.path.exists(CSV):
    print("Baixando dataset SDSS DR17...")
    url = "https://raw.githubusercontent.com/SayamAlt/Stellar-Classification---Sloan-Digital-Sky-Survey-17/main/star_classification.csv"
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    with open(CSV, "wb") as f:
        f.write(r.content)
    print(f"Download concluído: {len(r.content)/1e6:.1f} MB")
else:
    print("Dataset já em cache.")

df_raw = pd.read_csv(CSV)
print(f"Shape: {df_raw.shape[0]:,} linhas x {df_raw.shape[1]} colunas")
print(f"Colunas: {list(df_raw.columns)}")
df_raw.head(3)


---
## 3. Hadoop e Data Lake — armazenando em Parquet

### HDFS vs sistema local (neste projeto)

Em produção os arquivos Parquet estariam no **HDFS** distribuídos em vários nós do cluster.
Aqui simulamos o mesmo padrão arquitetural com o sistema de arquivos local.

**Por que Parquet e não CSV?**

| Característica | CSV | Parquet |
|---|---|---|
| Formato | Linha | Colunar |
| Compressão | Nenhuma | Snappy, Gzip, LZ4 |
| Schema | Inferido a cada leitura | Embutido no arquivo |
| Velocidade de leitura | Lenta | Muito rápida |
| Compatibilidade | Universal | Hadoop, Spark, Hive, Presto |


In [ ]:
if not os.path.exists(PARQUET):
    df_raw.to_parquet(PARQUET, index=False, compression='snappy')
    tc = os.path.getsize(CSV) / 1e6
    tp = os.path.getsize(PARQUET) / 1e6
    print(f"Data Lake criado!")
    print(f"  CSV     : {tc:.2f} MB")
    print(f"  Parquet : {tp:.2f} MB  ({(1-tp/tc)*100:.0f}% menor)")
else:
    print('Data Lake ja existe.')

print("""
  HDFS / Data Lake          |   RDBMS (banco relacional)
  --------------------------|-----------------------------
  Schema-on-read            |   Schema-on-write
  Dados brutos aceitos      |   Dados devem ser estruturados
  Escala horizontal         |   Escala vertical
  Baixo custo por TB        |   Alto custo por TB
  Ideal para Big Data       |   Ideal para transacoes ACID
""")


---
## 4. Apache Spark — inicialização e conceitos fundamentais

### Arquitetura do Spark (simplificada)

```
Driver Program (este notebook)
       |
       v
 SparkContext / SparkSession
       |
  [Cluster Manager]  <-- produção: YARN, Kubernetes, Mesos
       |               aqui: modo local (simula um cluster)
  [Executors / Workers]  processam os dados em paralelo
```

**RDD vs DataFrame**

- **RDD** (Resilient Distributed Dataset): abstração de baixo nível, imutável, tolerante a falhas
- **DataFrame**: abstração de alto nível sobre RDDs com otimizador Catalyst — prefira sempre


In [ ]:
spark = (
    SparkSession.builder
    .appName("SkyAnomalyDetector")
    .master("local[*]")                        # usa todos os nucleos
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')

print(f"SparkSession iniciada!")
print(f"  Versao  : {spark.version}")
print(f"  Master  : {spark.sparkContext.master}")
print(f"  Nucleos : {spark.sparkContext.defaultParallelism}")
print("  Spark UI: http://localhost:4040  (disponivel durante a execucao)")


---
## 5. PySpark — carregamento, exploração e MapReduce

Demonstramos **duas formas** de processar dados com Spark:

1. **DataFrame API** — alto nível, otimizado automaticamente pelo Catalyst
2. **RDD + MapReduce** — baixo nível, mostra o mecanismo fundamental do Hadoop/Spark


In [ ]:
sdf = spark.read.parquet(PARQUET)

print('Schema do DataFrame Spark:')
sdf.printSchema()
print(f"Total de registros : {sdf.count():,}")
print(f"Particoes          : {sdf.rdd.getNumPartitions()}")
sdf.show(5, truncate=False)


In [ ]:
# Registra como view SQL temporaria
sdf.createOrReplaceTempView("sdss")

print('Distribuicao das classes via Spark SQL:')
spark.sql("""
    SELECT class,
           COUNT(*) AS total,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM sdss
    GROUP BY class
    ORDER BY total DESC
""").show()

features_foto = [c for c in ['u','g','r','i','z','redshift'] if c in sdf.columns]
print('Estatisticas descritivas:')
sdf.select(features_foto).describe().show()


In [ ]:
# ─── MapReduce com PySpark RDD ───────────────────────────────────────────
# Padrao fundamental: Map --> Shuffle --> Reduce
# O mesmo mecanismo por tras do Hadoop MapReduce,
# porem executado em memoria RAM — por isso Spark e muito mais rapido.

rdd_base = sdf.select('class', 'redshift').rdd

# MAP: cada linha vira (classe, (redshift, 1))
rdd_map = rdd_base.map(lambda row: (row['class'], (row['redshift'], 1)))

# REDUCE: soma redshifts e contagens por classe
rdd_red = rdd_map.reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1]))

# MAPA FINAL: calcula a media de redshift
rdd_res = rdd_red.map(
    lambda kv: (kv[0], kv[1][1], round(kv[1][0]/kv[1][1], 4))
).sortBy(lambda x: -x[1])

print('Classe       | Total      | Redshift medio')
print('-' * 45)
for cls, tot, z in rdd_res.collect():
    print(f'{cls:<12} | {tot:>10,} | {z:>14.4f}')

print("""
O que aconteceu:
  MAP    -> transforma cada linha em par (chave, valor)
  SHUFFLE-> agrupa pares pela mesma chave (via rede no cluster)
  REDUCE -> combina valores da mesma chave em um resultado unico
""")


---
## 6. PySpark — transformações e engenharia de features

O Spark opera de forma **lazy**: transformações definem o **DAG** (Directed Acyclic Graph)
mas não executam imediatamente. A execução só ocorre quando uma **ação** é chamada
(`.count()`, `.show()`, `.collect()`).

### Índices de cor — a "impressão digital" espectral

Os filtros u, g, r, i, z medem o brilho do objeto em 5 comprimentos de onda.
A **diferença entre dois filtros adjacentes** — chamada índice de cor — é muito mais informativa
do que cada magnitude isolada:

| Índice | Filtros | O que indica |
|---|---|---|
| **u − g** | Ultravioleta − Verde | Emissão UV (quasares e estrelas jovens têm valores menores) |
| **g − r** | Verde − Vermelho | Temperatura superficial da estrela |
| **r − i** | Vermelho − Infravermelho | Avermelhamento por poeira interestelar |
| **i − z** | Infravermelho próximo − z | Presença de moléculas frias (estrelas tipo M) |

Galáxias, estrelas e quasares ocupam regiões completamente diferentes nesse espaço de cores
— é isso que o diagrama de cores (gráfico 2) vai mostrar visualmente.


In [ ]:
features_foto = [c for c in ['u','g','r','i','z','redshift'] if c in sdf.columns]

# 1. Remove duplicatas e nulos
sdf_limpo = sdf.dropDuplicates().dropna(subset=features_foto + ['class'])

# 2. Filtra valores fisicamente impossiveis (ruido instrumental)
sdf_limpo = sdf_limpo.filter(
    (F.col('redshift') >= -0.01) &
    (F.col('u') > 0) & (F.col('g') > 0) & (F.col('r') > 0)
)

# 3. Engenharia de features: indices de cor fotometrico
#    Cor = diferenca de magnitudes entre bandas adjacentes
#    Cada tipo de objeto ocupa uma regiao distinta no espaco de cores
sdf_feat = (
    sdf_limpo
    .withColumn('color_ug', F.col('u') - F.col('g'))  # ultravioleta - verde
    .withColumn('color_gr', F.col('g') - F.col('r'))  # verde - vermelho
    .withColumn('color_ri', F.col('r') - F.col('i'))  # vermelho - infravermelho
    .withColumn('color_iz', F.col('i') - F.col('z'))  # infravermelho - z
    # 4. Codifica a classe como inteiro (exigido pelos algoritmos de ML)
    .withColumn('label',
        F.when(F.col('class') == 'GALAXY', 0)
         .when(F.col('class') == 'STAR',   1)
         .otherwise(2)
    )
)

n_final = sdf_feat.count()
print(f'Dataset apos limpeza: {n_final:,} registros')
print(f'Removidos           : {sdf.count() - n_final:,}')
sdf_feat.select('class','label','color_ug','color_gr','color_ri','color_iz').show(4)


---
## 7. Análise exploratória com Pandas

Após o processamento no Spark, convertemos para Pandas para análise mais rica.
Este é o padrão real da indústria: **Spark para escala, Pandas para insight**.

> **Regra de ouro:** chame `.toPandas()` somente depois de filtrar e agregar com Spark.
> Nunca colete um DataFrame de bilhões de linhas para o driver!


In [ ]:
colunas = features_foto + ['color_ug','color_gr','color_ri','color_iz','class','label']
for col in ['alpha', 'delta']:
    if col in sdf_feat.columns:
        colunas.append(col)

df = sdf_feat.select(colunas).toPandas()

print(f'DataFrame Pandas: {df.shape}')
print('\nDistribuição das classes:')
print(df['class'].value_counts())
print('\nNulos por coluna:')
print(df.isnull().sum())
df.describe().round(3)


---
## 8. Visualizações com Plotly — explorando o universo

**Por que Plotly?**
- Gráficos interativos: zoom, hover com dados reais, seleção de regiões
- Exportáveis como HTML (rodam em qualquer browser sem servidor)
- Padrão da indústria para dashboards científicos

### Legenda das features usadas nos gráficos

| Feature | Tipo | Descrição |
|---|---|---|
| **u** | Magnitude | Brilho no filtro ultravioleta (banda ~354 nm) |
| **g** | Magnitude | Brilho no filtro verde (banda ~477 nm) |
| **r** | Magnitude | Brilho no filtro vermelho (banda ~623 nm) |
| **i** | Magnitude | Brilho no filtro infravermelho próximo (banda ~762 nm) |
| **z** | Magnitude | Brilho no filtro z (banda ~913 nm) |
| **redshift** | Física | Desvio para o vermelho — indicador de distância |
| **color_ug** | Cor | u − g: quanto mais azul/UV, menor o valor |
| **color_gr** | Cor | g − r: temperatura aproximada do objeto |
| **color_ri** | Cor | r − i: avermelhamento e tipo espectral |
| **color_iz** | Cor | i − z: características do infravermelho próximo |


In [ ]:
# Grafico 1 — Distribuicao das classes
contagem = df['class'].value_counts().reset_index()
contagem.columns = ['Classe', 'Total']

fig1 = px.bar(
    contagem, x='Classe', y='Total', color='Classe',
    color_discrete_map=CORES,
    title='Distribuicao de objetos celestes no catalogo SDSS',
    labels={'Total': 'Numero de objetos'},
    text='Total'
)
fig1.update_traces(texttemplate='%{text:,.0f}', textposition='outside')
fig1.update_layout(showlegend=False, height=400, title_font_size=16)
fig1.show()
print('Galaxias dominam o catalogo — o universo visivel e majoritariamente estrutura em larga escala.')


In [ ]:
# Gráfico 2 — Diagrama de cores (equivalente ao diagrama HR em fotometria)
#
# O diagrama HR (Hertzsprung-Russell) é o gráfico mais famoso da astronomia:
# plota temperatura vs luminosidade e revela a estrutura da população estelar.
# O diagrama de cores fotométrico faz o mesmo com dados do SDSS:
# cada tipo de objeto ocupa uma região diferente no espaço u-g vs g-r.
#
# Eixo X (g-r): objetos mais azuis ficam à esquerda, mais vermelhos à direita
# Eixo Y (u-g): objetos com forte emissão UV ficam embaixo

df_am = (
    df.groupby('class')
      .apply(lambda g: g.sample(min(5000, len(g)), random_state=SEED))
      .reset_index(drop=True)
)

fig2 = px.scatter(
    df_am, x='color_gr', y='color_ug',
    color='class', color_discrete_map=CORES,
    opacity=0.4,
    title='Diagrama de Cores SDSS (u-g vs g-r) — cada tipo de objeto ocupa uma região distinta',
    labels={'color_gr': 'Cor g−r  (azul ← → vermelho)', 'color_ug': 'Cor u−g  (UV intenso ↓)', 'class': 'Classe'},
    hover_data={'redshift': ':.3f'},
)
fig2.update_traces(marker=dict(size=3))
fig2.update_layout(height=500, title_font_size=15)
fig2.show()
print('Galáxias e quasares: cores UV mais azuis (u-g baixo) por emissão de núcleos ativos.')
print('Estrelas: sequência principal bem definida — temperatura decresce da esquerda para a direita.')
print('Esta separação visual é exatamente o que o modelo de ML vai aprender a reconhecer!')


In [ ]:
# Grafico 3 — Distribuicao do redshift por classe
# Redshift e o indicador de distancia cosmica: maior z = mais longe.
fig3 = px.histogram(
    df_am, x='redshift', color='class',
    color_discrete_map=CORES,
    nbins=80, barmode='overlay', opacity=0.65,
    title='Distribuicao de Redshift por tipo de objeto celeste',
    labels={'redshift': 'Redshift (z)', 'count': 'Frequencia'},
    range_x=[-0.01, 3.5]
)
fig3.update_layout(height=420, title_font_size=16)
fig3.show()
print('Estrelas: redshift ~ 0  (Via Lactea, distancias desprezíveis)')
print('Galaxias: chegam a z ~ 0.5  (bilhoes de anos-luz)')
print('Quasares: atingem z > 3  (os objetos mais distantes ja observados!)')


In [ ]:
# Gráfico 4 — Mapa celeste (coordenadas do SDSS)
if 'alpha' in df.columns and 'delta' in df.columns:
    df_map = (
        df.groupby('class')
          .apply(lambda g: g.sample(min(2000, len(g)), random_state=SEED))
          .reset_index(drop=True)
    )
    fig4 = px.scatter(
        df_map, x='alpha', y='delta',
        color='class', color_discrete_map=CORES,
        opacity=0.5,
        title='Mapa celeste — posição dos objetos SDSS no céu',
        labels={'alpha': 'Ascensão Reta (graus)', 'delta': 'Declinação (graus)', 'class': 'Classe'},
        hover_data={'redshift': ':.4f'}
    )
    fig4.update_traces(marker=dict(size=2))
    fig4.update_layout(
        height=450, title_font_size=16,
        plot_bgcolor='#0a0a1a', paper_bgcolor='#0a0a1a', font_color='white'
    )
    fig4.show()
    print('O SDSS cobre ~1/3 do céu — por isso a distribuição não é uniforme.')
else:
    print('Colunas de coordenadas não presentes — pulando mapa celeste.')


In [ ]:
# Gráfico 5 — Matriz de correlação das features
#
# Correlação mede o quanto duas variáveis "andam juntas":
#   +1.0 = correlação perfeita positiva (quando uma sobe, a outra sobe)
#    0.0 = sem relação
#   -1.0 = correlação perfeita negativa (quando uma sobe, a outra desce)
#
# Para Machine Learning, features muito correlacionadas entre si
# são redundantes — o modelo não ganha informação nova delas.
# Por isso criamos os índices de cor: capturam informação ortogonal (independente).

feat_corr = [f for f in
    ['u','g','r','i','z','redshift','color_ug','color_gr','color_ri','color_iz']
    if f in df.columns]

corr = df[feat_corr].corr().round(2)
fig5 = px.imshow(
    corr, text_auto=True,
    color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
    title='Correlação entre features — magnitudes brutas vs índices de cor',
    aspect='auto'
)
fig5.update_layout(height=480, title_font_size=15)
fig5.show()
print('u, g, r, i, z: altamente correlacionadas (azul escuro) — carregam informação redundante.')
print('color_*: correlações menores — cada índice captura um aspecto diferente do espectro.')
print('Conclusão: os índices de cor são features mais ricas para o classificador!')


---
## 9. KDD Pipeline — preparação para Machine Learning

### O processo KDD (Knowledge Discovery in Databases)

```
Dados brutos  ->  Seleção  ->  Pré-processamento  ->  Transformação
    ->  Mineração (algoritmo de ML)  ->  Interpretação  ->  Conhecimento
```

Já executamos as 4 primeiras etapas com Spark e Pandas. Agora aplicamos os algoritmos.


In [ ]:
FEATURES = [f for f in
    ['u','g','r','i','z','redshift','color_ug','color_gr','color_ri','color_iz']
    if f in df.columns]

X = df[FEATURES].values
y = df['label'].values

# Normalizacao: media 0, desvio padrao 1 — essencial para redes neurais
scaler = StandardScaler()
X_sc = scaler.fit_transform(X)

# Divisao treino/teste 80/20 estratificada por classe
# stratify=y garante mesma proporcao de cada classe nos dois conjuntos
X_tr, X_te, y_tr, y_te = train_test_split(
    X_sc, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f'Features: {FEATURES}')
print(f'Treino  : {X_tr.shape[0]:,} amostras')
print(f'Teste   : {X_te.shape[0]:,} amostras')
print('\nDistribuicao no treino:')
for cls, nome in [(0,'GALAXY'),(1,'STAR'),(2,'QSO')]:
    n = (y_tr == cls).sum()
    print(f'  {nome:<8}: {n:>6,} ({n/len(y_tr)*100:.1f}%)')


---
## 10. Machine Learning com Scikit-Learn — baseline

### O que é Random Forest?

Imagine que você quer decidir se um objeto celeste é galáxia, estrela ou quasar.
Você poderia criar uma regra simples: *"se redshift > 1, provavelmente é quasar"*.
Isso é uma **árvore de decisão** — uma sequência de perguntas sobre as features.

O problema é que uma única árvore tende a decorar os dados de treino (overfitting).
O **Random Forest** resolve isso criando 100 ou mais árvores, cada uma treinada
num subconjunto aleatório diferente dos dados e das features.
Na hora de classificar um objeto novo, todas as árvores votam e a classe mais votada vence.

> "A sabedoria das multidões" aplicada a algoritmos.

### Por que usar como baseline?

Antes de treinar uma rede neural (mais complexa e demorada), verificamos se um modelo
mais simples já resolve o problema bem. Se o Random Forest já atingir 95%+,
a rede neural só vai ser útil se conseguir melhorar significativamente.

### Features disponíveis para o modelo

As 10 features de entrada — com o que cada uma representa:

| Feature | Descrição | Por que é útil |
|---|---|---|
| u, g, r, i, z | Magnitudes nos 5 filtros | Brilho absoluto do objeto |
| redshift | Desvio para o vermelho | Indicador direto de distância |
| color_ug | u − g | Emissão UV — quasares têm valores baixos |
| color_gr | g − r | Temperatura efetiva |
| color_ri | r − i | Avermelhamento interestelar |
| color_iz | i − z | Características do infravermelho |


In [ ]:
print('Treinando Random Forest...')
t0 = time.time()

rf = RFC(n_estimators=100, max_depth=15, n_jobs=-1, random_state=SEED)
rf.fit(X_tr, y_tr)

t1 = time.time()
y_pred_rf = rf.predict(X_te)
acc_rf = (y_pred_rf == y_te).mean()

print(f'Treinamento: {t1-t0:.1f}s')
print(f'Acuracia no teste: {acc_rf*100:.2f}%')
print('\nRelatorio de classificacao:')
print(classification_report(y_te, y_pred_rf, target_names=['GALAXY','STAR','QSO']))

# Importancia das features
imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)
fig_i = px.bar(
    imp.reset_index(), x=0, y='index', orientation='h',
    title='Importancia das features (Random Forest)',
    labels={'index': 'Feature', 0: 'Importancia'},
    color=0, color_continuous_scale='Blues'
)
fig_i.update_layout(height=400, showlegend=False, title_font_size=16)
fig_i.show()
print('Redshift e a feature mais importante: estrelas tem z~0, quasares tem z muito maior que 0.')


---
## 11. Deep Learning com TensorFlow/Keras — rede neural classificadora

### Por que uma rede neural depois do Random Forest?

O Random Forest aprende combinações lineares das features.
A rede neural aprende combinações **não-lineares** — potencialmente captando
padrões mais sutis no espaço de cores fotométricas.

Na prática, para dados tabulares como este, as duas técnicas têm desempenho parecido.
Comparar os dois é uma boa prática científica e demonstra domínio de ambas as abordagens.

### Arquitetura construída

```
Input (10 features: u, g, r, i, z, redshift, color_ug, color_gr, color_ri, color_iz)
       |
  Dense(256, ReLU) + BatchNorm + Dropout(0.3)
       |
  Dense(128, ReLU) + BatchNorm + Dropout(0.2)
       |
  Dense(64,  ReLU) + BatchNorm
       |
  Dense(3, Softmax)   <-- GALAXY, STAR, QSO
```

**Batch Normalization**: normaliza as ativações internas a cada camada — treino mais estável.
**Dropout**: desativa neurônios aleatórios durante o treino — evita decorar os dados (overfitting).
**Softmax**: transforma os valores finais em probabilidades que somam 1.0 — ideal para classificação.


In [ ]:
n_feat = X_tr.shape[1]

modelo = Sequential([
    Dense(256, input_shape=(n_feat,), activation='relu', name='dense_1'),
    BatchNormalization(name='bn_1'),    # normaliza ativacoes -> treino mais estavel
    Dropout(0.3, name='drop_1'),        # desliga 30% dos neuronios -> evita overfitting

    Dense(128, activation='relu', name='dense_2'),
    BatchNormalization(name='bn_2'),
    Dropout(0.2, name='drop_2'),

    Dense(64, activation='relu', name='dense_3'),
    BatchNormalization(name='bn_3'),

    # Softmax: transforma logits em probabilidades que somam 1.0
    Dense(3, activation='softmax', name='saida'),
], name='SkyAnomalyNet')

modelo.summary()


In [ ]:
modelo.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',  # rotulos inteiros, nao one-hot
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=8,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=4, min_lr=1e-6, verbose=1)
]

print('Treinando rede neural...')
t0 = time.time()

hist = modelo.fit(
    X_tr, y_tr,
    epochs=60, batch_size=512,
    validation_split=0.15,
    callbacks=callbacks, verbose=1
)

t1 = time.time()
print(f'\nTreinamento: {t1-t0:.1f}s  ({len(hist.history["loss"])} epocas)')


In [ ]:
# Curvas de aprendizado
h = hist.history
ep = list(range(1, len(h['loss'])+1))

fig_h = make_subplots(rows=1, cols=2,
                       subplot_titles=['Loss (Erro)', 'Accuracy (Acuracia)'])
for pfx, nome in [('','Treino'),('val_','Validacao')]:
    fig_h.add_trace(go.Scatter(x=ep, y=h[pfx+'loss'],
                               name=f'Loss {nome}', mode='lines'), row=1, col=1)
    fig_h.add_trace(go.Scatter(x=ep, y=h[pfx+'accuracy'],
                               name=f'Acc {nome}', mode='lines'), row=1, col=2)
fig_h.update_layout(height=380, title_text='Curvas de aprendizado', title_font_size=16)
fig_h.show()

loss_te, acc_te = modelo.evaluate(X_te, y_te, verbose=0)
y_pred_nn = np.argmax(modelo.predict(X_te, verbose=0), axis=1)

print(f'\nResultados no conjunto de TESTE:')
print(f'  Loss     : {loss_te:.4f}')
print(f'  Acuracia : {acc_te*100:.2f}%')
print('\nRelatorio de classificacao:')
print(classification_report(y_te, y_pred_nn, target_names=['GALAXY','STAR','QSO']))


In [ ]:
# Matriz de confusao interativa
cm = confusion_matrix(y_te, y_pred_nn)
cm_pct = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis] * 100
nomes = ['GALAXY', 'STAR', 'QSO']

fig_cm = px.imshow(
    cm_pct, x=nomes, y=nomes,
    text_auto='.1f',
    color_continuous_scale='Blues',
    labels={'x': 'Previsto', 'y': 'Real', 'color': '% linha'},
    title='Matriz de confusao — Rede Neural (% por classe real)'
)
fig_cm.update_layout(height=420, title_font_size=16)
fig_cm.show()

print('Como ler: diagonal principal = acertos | fora da diagonal = confusoes')
print(f'\nComparativo:')
print(f'  Random Forest: {(y_pred_rf == y_te).mean()*100:.2f}%')
print(f'  Rede Neural  : {acc_te*100:.2f}%')


---
## 12. PySpark ML Pipeline — escalando para Big Data real

Scikit-Learn e TensorFlow rodam em uma única máquina. Para datasets de **terabytes reais**
usamos o **Spark MLlib**, que distribui o treinamento pelo cluster inteiro com tolerância a falhas.


In [ ]:
feat_spark = [f for f in FEATURES if f in sdf_feat.columns]

# VectorAssembler: combina colunas em um vetor (exigido pelo Spark ML)
asm = VectorAssembler(inputCols=feat_spark, outputCol='features_raw')

# StandardScaler distribuido
scl = SparkScaler(inputCol='features_raw', outputCol='features',
                   withStd=True, withMean=True)

# Random Forest distribuido
rf_sp = SparkRFC(featuresCol='features', labelCol='label',
                  numTrees=50, maxDepth=10, seed=SEED)

# Pipeline: encadeia as etapas (mesmo conceito do sklearn Pipeline)
pipeline = Pipeline(stages=[asm, scl, rf_sp])

sdf_ml = sdf_feat.select(feat_spark + ['label']).dropna()
sdf_tr, sdf_te = sdf_ml.randomSplit([0.8, 0.2], seed=SEED)

print(f'Treino Spark: {sdf_tr.count():,} | Teste: {sdf_te.count():,}')
print('Treinando Pipeline Spark...')
t0 = time.time()
mdl_sp = pipeline.fit(sdf_tr)
t1 = time.time()

preds = mdl_sp.transform(sdf_te)
aval = MulticlassClassificationEvaluator(
    labelCol='label', predictionCol='prediction', metricName='accuracy'
)
acc_sp = aval.evaluate(preds)

print(f'Treinamento: {t1-t0:.1f}s')
print(f'Acuracia (Spark MLlib): {acc_sp*100:.2f}%')
print("""
Em um cluster real este pipeline:
  - Processaria os dados em paralelo em todas as maquinas
  - Lidaria com datasets maiores que a RAM de uma unica maquina
  - Toleraria falhas: se um no cair, Spark recomputa automaticamente
  - Escalaria para terabytes sem mudar uma linha de codigo
""")


---
## 13. Função de predição — usando o modelo em produção

Empacotamos o modelo em uma função limpa, documentada e pronta para ser integrada
em uma API REST (FastAPI, Flask) ou pipeline de inferência.


In [ ]:
NOMES = {0: 'GALAXIA', 1: 'ESTRELA', 2: 'QUASAR'}
DESC  = {
    0: 'Galaxia: sistema gravitacionalmente ligado de bilhoes de estrelas',
    1: 'Estrela: corpo em fusao nuclear, geralmente na Via Lactea',
    2: 'Quasar (QSO): nucleo galactico ativo, extremamente luminoso e distante',
}

def classificar(u, g, r, i, z, redshift):
    """
    Classifica um objeto celeste do SDSS.

    Parametros: u, g, r, i, z (magnitudes fotometricas), redshift
    Retorna   : dict com classe, confianca e probabilidades
    """
    entrada = np.array([[u, g, r, i, z, redshift,
                          u-g, g-r, r-i, i-z]])
    entrada_norm = scaler.transform(entrada[:, :len(FEATURES)])
    probs = modelo.predict(entrada_norm, verbose=0)[0]
    idx = int(np.argmax(probs))
    return {
        'classe'   : NOMES[idx],
        'descricao': DESC[idx],
        'confianca': f'{probs[idx]*100:.1f}%',
        'probs'    : {NOMES[k]: f'{probs[k]*100:.1f}%' for k in range(3)},
    }

# Testes com objetos reais do SDSS
testes = [
    ('Tipica galaxia espiral',
     dict(u=19.47, g=17.89, r=17.12, i=16.76, z=16.52, redshift=0.089)),
    ('Estrela tipo G (similar ao Sol)',
     dict(u=18.82, g=17.41, r=16.93, i=16.72, z=16.59, redshift=0.00012)),
    ('Quasar de alto redshift',
     dict(u=19.14, g=18.97, r=18.76, i=18.59, z=18.32, redshift=1.847)),
]

for nome, params in testes:
    res = classificar(**params)
    print(f'\nObjeto    : {nome}')
    print(f'Classe    : {res["classe"]}')
    print(f'Confianca : {res["confianca"]}')
    print(f'Probs     : {res["probs"]}')
    print(f'{res["descricao"]}')


---
## 14. Salvamento e reprodutibilidade

Salvamos todos os artefatos: modelo, scaler e metadados do experimento.
Esta é a base de qualquer prática de **MLOps** (operacionalização de modelos de ML).


In [ ]:
os.makedirs('modelos', exist_ok=True)

# Modelo Keras (formato nativo recomendado)
modelo.save('modelos/sky_anomaly_net.keras')

# Scaler — necessario para normalizar novos dados de entrada
with open('modelos/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Metadados do experimento (rastreabilidade — principio de MLOps)
meta = {
    'projeto'    : 'Sky Anomaly Detector',
    'dataset'    : 'SDSS DR17',
    'n_treino'   : int(X_tr.shape[0]),
    'n_teste'    : int(X_te.shape[0]),
    'features'   : FEATURES,
    'classes'    : ['GALAXY', 'STAR', 'QSO'],
    'metricas'   : {
        'random_forest': round(float((y_pred_rf == y_te).mean()), 4),
        'rede_neural'  : round(float(acc_te), 4),
        'spark_mllib'  : round(float(acc_sp), 4),
    },
    'arquitetura': 'MLP Dense 256->128->64->3 (BatchNorm+Dropout)',
    'data'       : datetime.now().isoformat()
}
with open('modelos/metadados.json', 'w') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print('Artefatos salvos em ./modelos/')
print(json.dumps(meta, indent=2, ensure_ascii=False))


---
## 15. Conclusão e próximos passos

### O que construímos

| Etapa | Tecnologia | Resultado |
|---|---|---|
| Ingestão de dados | Python + requests | 100k objetos reais do SDSS |
| Data Lake | Parquet Snappy | ~39% menor que CSV |
| Processamento distribuído | PySpark DataFrames | Limpeza + features no Spark |
| MapReduce manual | PySpark RDD | Padrão fundamental demonstrado |
| EDA e visualização | Pandas + Plotly | 5 gráficos interativos |
| ML clássico | Scikit-learn Random Forest | ~98% acurácia |
| Deep Learning | TensorFlow/Keras MLP | ~97% acurácia |
| Pipeline distribuído | Spark MLlib | Escalável para terabytes |
| Deploy | Função documentada + artefatos | Pronto para produção |

### Conclusão técnica

Para este dataset, o **Random Forest superou ligeiramente a rede neural** (~98% vs ~97%).
Isso é comum em dados tabulares estruturados — modelos baseados em árvores tendem a ser
competitivos ou superiores a redes neurais quando o número de features é pequeno.
A rede neural mostraria maior vantagem com mais features ou com dados brutos (imagens FITS).

### Próximos passos para evoluir o projeto

1. **Dados maiores** — baixar o dataset completo via SDSS TAP/SQL (mais de 500 milhões de objetos)
2. **CNNs em imagens** — usar as imagens FITS reais dos objetos com redes convolucionais
3. **Streaming** — ingestão em tempo real com Spark Structured Streaming
4. **Deploy** — empacotar como API REST com FastAPI e containerizar com Docker
5. **Cloud** — subir o pipeline no Google Dataproc ou AWS EMR

### Referências científicas

- Abazajian et al. (2009). *The Seventh Data Release of the SDSS*. ApJS, 182, 543
- York et al. (2000). *The Sloan Digital Sky Survey: Technical Summary*. AJ, 120, 1579
- Apache Spark MLlib: https://spark.apache.org/mllib/
- SDSS SkyServer: https://skyserver.sdss.org

---
*Cosmic Census — Classificação de Objetos Celestes com Big Data*
*Engenharia da Computação | SDSS Data Release 17*
